# 07 — Explain and rank high-risk test cases

This notebook turns the saved LightGBM scores into a review queue and shows context around selected accounts.

The queue is for human review. A high score is a signal to investigate, not proof of fraud. The test labels are used only for retrospective evaluation; the review queue intentionally does not include the answer label.

In [ ]:
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd

project_root = Path.cwd()
feature_path = project_root / "data" / "processed" / "paysim_graph_features.parquet"
model_path = project_root / "models" / "fraud_risk_model.joblib"
if not feature_path.exists() or not model_path.exists():
    raise FileNotFoundError("Run notebook 05_full_data_features.ipynb and 06_model_baselines.ipynb first.")

artifact = joblib.load(model_path)
model = artifact["model"]
feature_columns = artifact["feature_columns"]
encoded_columns = artifact["encoded_columns"]
threshold = artifact["threshold"]

feature_sql = feature_path.resolve().as_posix().replace("'", "''")
db = duckdb.connect(database=":memory:")
test_cases = db.execute(f"""
    SELECT step, type, amount, oldbalanceOrg, oldbalanceDest,
           prior_outgoing_count, prior_outgoing_amount,
           prior_incoming_count, prior_incoming_amount,
           nameOrig, nameDest, isFraud
    FROM read_parquet('{feature_sql}')
    WHERE data_split = 'test'
""").df()

encoded = pd.get_dummies(test_cases[feature_columns], columns=["type"], dtype="float32")
encoded = encoded.reindex(columns=encoded_columns, fill_value=0).astype("float32")
test_cases["risk_score"] = model.predict_proba(encoded)[:, 1]
test_cases["encoded_row_index"] = encoded.index
test_cases["alert_candidate"] = test_cases["risk_score"] >= threshold
test_cases = test_cases.sort_values("risk_score", ascending=False).reset_index(drop=True)
test_cases["risk_rank"] = np.arange(1, len(test_cases) + 1)

print(f"Test transactions ranked: {len(test_cases):,}")
print(f"Validation-selected alert threshold: {threshold:.4f}")
print(f"Alert candidates in test period: {int(test_cases['alert_candidate'].sum()):,}")
print("The label is retained here only to measure performance after scoring.")

## Review queue

The top-ranked cases are the transactions with the largest model risk scores. We keep the actual test label out of the queue, just as an analyst would not know the answer before investigation.

In [ ]:
review_columns = [
    "risk_rank", "risk_score", "alert_candidate", "step", "type", "amount",
    "nameOrig", "nameDest",
    "prior_outgoing_count", "prior_incoming_count",
]
review_queue = test_cases.head(500)[review_columns].copy()
evaluation_view = test_cases.head(500)[review_columns + ["isFraud"]].rename(
    columns={"isFraud": "actual_label_for_evaluation"}
)
display(review_queue.head(20))
print(f"Fraud-labeled cases among the top 500 (evaluation only): {int(evaluation_view['actual_label_for_evaluation'].sum())}")

## Local model explanation

LightGBM can provide additive feature contributions for each scored transaction. Positive contributions push the model score upward; negative contributions push it downward. These contributions explain the model's calculation, not the cause of a real-world event.

In [ ]:
top_for_explanation = test_cases.head(5)
X_explanation = encoded.loc[top_for_explanation["encoded_row_index"].tolist()]
contributions = model.booster_.predict(X_explanation, pred_contrib=True)
contribution_names = encoded_columns + ["baseline_contribution"]

explanation_rows = []
for row_position, (_, case) in enumerate(top_for_explanation.iterrows()):
    values = contributions[row_position][:-1]
    strongest = np.argsort(values)[::-1][:4]
    for feature_index in strongest:
        explanation_rows.append({
            "risk_rank": int(case["risk_rank"]),
            "risk_score": float(case["risk_score"]),
            "feature": contribution_names[feature_index],
            "contribution": float(values[feature_index]),
            "direction": "raises score" if values[feature_index] > 0 else "lowers score",
        })

local_explanations = pd.DataFrame(explanation_rows)
display(local_explanations)

## Earlier account neighborhood

For the highest-ranked cases, inspect earlier transfers involving the sender and receiver. Only transactions from steps before the case are shown. The fraud label is omitted from this neighborhood so it cannot be mistaken for information available to an investigator at scoring time.

In [ ]:
neighborhood_rows = []
for _, case in test_cases.head(3).iterrows():
    for role, account in [("sender", case["nameOrig"]), ("receiver", case["nameDest"])]:
        history = db.execute(f"""
            SELECT step, type, amount, nameOrig, nameDest
            FROM read_parquet('{feature_sql}')
            WHERE step < ? AND (nameOrig = ? OR nameDest = ?)
            ORDER BY step DESC
            LIMIT 10
        """, [int(case["step"]), account, account]).df()
        if not history.empty:
            history["case_risk_rank"] = int(case["risk_rank"])
            history["focal_role"] = role
            history["focal_account"] = account
            neighborhood_rows.append(history)

if neighborhood_rows:
    neighborhoods = pd.concat(neighborhood_rows, ignore_index=True)
else:
    neighborhoods = pd.DataFrame(columns=[
        "step", "type", "amount", "nameOrig", "nameDest",
        "case_risk_rank", "focal_role", "focal_account",
    ])
display(neighborhoods.head(30))

## Save review outputs

The review queue omits the hidden test label. The evaluation file includes that label for project validation only; do not present it as an analyst input.

In [ ]:
reports_dir = project_root / "reports"
reports_dir.mkdir(exist_ok=True)
review_queue.to_csv(reports_dir / "investigation_queue.csv", index=False)
evaluation_view.to_csv(reports_dir / "top_500_evaluation.csv", index=False)
local_explanations.to_csv(reports_dir / "top_case_explanations.csv", index=False)
neighborhoods.to_csv(reports_dir / "top_case_neighborhoods.csv", index=False)
print(f"Saved review queue and explanation tables in {reports_dir}")

## What to tell a mentor

The model ranks transactions by risk. For a selected case, we show which model inputs raised or lowered its score and display earlier transfers connected to the sender and receiver. The analyst reviews that context; the score is not a fraud verdict.